# Parse Chemical Reaction Diagrams with NVIDIA Nemotron 3.5 Super VL

Fine-tune [Nemotron 3.5 Super VL (BF16)](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-BF16) with LoRA to turn reaction
diagrams into JSON describing reactants, conditions, and products.

We use the approach from [RxnCaption](https://arxiv.org/abs/2511.02384): draw numbered
boxes around molecules, then ask the model to describe the reactions. The notebook
downloads the pinned public
[U-RxnDiagram-15k dataset](https://huggingface.co/datasets/opendatalab/U-RxnDiagram-15k)
and creates the numbered images and JSON records automatically.

- **Requirements:**
  - Linux on ARM64 with Docker and NVIDIA Container Toolkit.
  - One node with four GB200 GPUs.
  - Git, outbound access to Hugging Face, GitHub, NGC, and Docker Hub.
  - About 1 TiB of free disk space.
  - At least 64 GiB of host RAM for merging; training also needs substantial host RAM.
- **Files:**
  - Keep this notebook, `rxncaption_support.py`, `prepare_rxncaption_data.py`,
    `reaction_example.png`, and `DATA_LICENSE.md` together.
  - Run the cells in order.
- **Inputs:**
  - The public model, dataset, AutoModel source, and ARM64 containers are preconfigured
    with immutable revisions/digests from the completed run.
  - Environment variables in Step 2 are optional explicit overrides.
- **Data license:**
  - U-RxnDiagram-15k and its converted artifacts are CC BY-NC 4.0. Attribution is required;
    commercial use is prohibited without separate permission from the dataset rights holder.
- **First run:**
  - The default `SMOKE_STEPS = 5` runs a short end-to-end check. Set it to `None`
    only when you intend to train for one complete epoch.
  - Both modes load the full model and evaluate all 400 images.


## Agenda

1. [Install dependencies](#step-1-install-dependencies)
2. [Configure the run](#step-2-configure-the-run)
3. [Download the model and data](#step-3-download-the-model-and-data)
4. [Prepare the data](#step-4-prepare-the-data)
5. [Set up LoRA training](#step-5-set-up-lora-training)
6. [Train on four GPUs](#step-6-train-on-four-gpus)
7. [Merge the adapter](#step-7-merge-the-adapter)
8. [Set up evaluation](#step-8-set-up-evaluation)
9. [Evaluate the model](#step-9-evaluate-the-model)
10. [Read the results](#step-10-read-the-results)


## A training example

The model receives this image and a prompt. Its target is the annotated reaction
JSON: `mol` refers to a numbered box, `txt` to text, and `idt` to a compound label.

![Chemical reaction diagram with numbered molecular boxes](reaction_example.png)


**Prompt excerpt:**

> Please extract the list of reactions from the image. Return only the JSON reaction list without adding any other content.

**Dataset target** (annotations can omit visible details):

```json
[
  {
    "reactants": [{"type": "mol", "index": 1}, {"type": "mol", "index": 2}],
    "conditions": [{"type": "txt", "content": "base (2.0 equiv) solvent, RT, 3 h"}],
    "products": [{"type": "mol", "index": 3}]
  }
]
```


## Step 1: Install dependencies

If you need Jupyter, run this in a terminal beside the notebook:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install jupyterlab ipykernel
jupyter lab chemical_reaction_parsing_lora.ipynb
```

The next cell installs the notebook packages. PyTorch, AutoModel, and vLLM run
in containers.


In [ ]:
%pip install "huggingface_hub>=0.34,<2" "PyYAML>=6,<7" "Pillow>=10.1,<13" "numpy>=1.26,<3"

In [ ]:
import json
import os
import re
from pathlib import Path

import yaml
from IPython.display import display
from PIL import Image as PILImage

# Local tutorial utilities; keep this file beside the notebook.
from rxncaption_support import (
    download,
    prepare_records,
    run_logged,
    sha256,
    write_json,
    write_runtime_scripts,
)

## Step 2: Configure the run

Choose a `WORKSPACE` for downloads and outputs and a new `RUN_ID` for each run.
Everything else has a reproducible default:

- Public GA BF16 model revision and public U-RxnDiagram-15k revision.
- AutoModel source commit `eb11d83ece9952b12e2f770df9fc524c1cbfe6fb`.
- ARM64 NeMo AutoModel 26.08 image digest used for training and merge.
- ARM64 vLLM nightly digest whose build commit
  `36768d1bfd39094681cdbc8cb37d4b31c0729c89` was used for evaluation.

The container digests are ARM64 because the completed run used GB200. Override
them only when deliberately porting the recipe to another architecture. Docker
mounts your workspace at `/workspace`; `TRAIN_PYTHON` and `EVAL_PYTHON` are the
executables verified inside those images.

The public GA model replaced the pre-release snapshot used for the reported
reference scores. Start with the five-step smoke test before a full epoch.


In [ ]:
WORKSPACE = Path("./super-vl-workspace").expanduser().resolve()
RUN_ID = "lora_rxn_01"
SMOKE_STEPS = 5  # use None only after this end-to-end smoke test passes
VALIDATION_SIZE = 400

CONFIG = {
    # Official BF16 GA release; pin verified on 2026-10-15.
    "model": {
        "repo_id": os.environ.get("SUPER_VL_MODEL_ID", "nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-BF16"),
        "revision": os.environ.get("SUPER_VL_MODEL_REVISION", "97e009758ae21747ee3752d2e43b005fb1e23a61"),
    },
    # Public source release converted locally into both training and benchmark inputs.
    "data": {
        "repo_id": os.environ.get("RXN_DATA_REPO_ID", "opendatalab/U-RxnDiagram-15k"),
        "revision": os.environ.get("RXN_DATA_REVISION", "274be957df104c4fde2104c2e02578b643aad3d1"),
        "train_images": 15_000,
        "test_images": 400,
    },
    "automodel": {
        "url": "https://github.com/NVIDIA-NeMo/Automodel.git",
        "revision": os.environ.get("AUTOMODEL_REVISION", "eb11d83ece9952b12e2f770df9fc524c1cbfe6fb"),
    },
    # Tested ARM64 NeMo AutoModel 26.08 image.
    "train_image": os.environ.get(
        "AUTOMODEL_IMAGE",
        "nvcr.io/nvidia/nemo-automodel@sha256:70b4e9df8dd23ef451372b25952cffe24ad29c17f5b9de5fd1b244caf4e7b55a",
    ),
    # Tested ARM64 nightly: vLLM 0.30.1rc1.dev312+g36768d1bf.
    "eval_image": os.environ.get(
        "VLLM_IMAGE",
        "vllm/vllm-openai@sha256:53377c87ef734ad0aa7eadc630605930c23b36ebc4785d161e05a7823a6a69d2",
    ),
}
TRAIN_PYTHON = "/opt/venv/bin/python"
EVAL_PYTHON = "/usr/bin/python3"
EVAL_PORT = 12520
CPU_MEMORY_NODES = ""  # optional CPU NUMA node list for your hardware
RUNTIME_ENV = {}  # optional container networking/runtime variables

assert re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,79}", RUN_ID), "Invalid run ID"
assert SMOKE_STEPS is None or 1 <= SMOKE_STEPS <= 20
assert "," not in str(WORKSPACE), "Docker bind-mount paths cannot contain commas"
ASSETS = WORKSPACE / "assets"
DATA = ASSETS / "data"
BENCHMARK = ASSETS / "benchmark"
BASE_MODEL = ASSETS / "model"
RUN_DIR = WORKSPACE / "runs" / RUN_ID
print("Downloads:", ASSETS)
print("Run outputs:", RUN_DIR)

## Step 3: Download the model, data, source, and containers

The next cell is now self-contained:

1. Download the pinned public U-RxnDiagram train/test archives.
2. Safely extract and convert all 15,400 images into numbered BIVP training and
   benchmark inputs with `prepare_rxncaption_data.py`.
3. Download the pinned BF16 model snapshot.
4. Check out the tested AutoModel source commit.
5. Pull the pinned ARM64 training and evaluation container digests.

Completed downloads and conversion outputs are identity-checked and reused.
Choose a new `WORKSPACE` rather than mixing different revisions. The generated
training and benchmark directories retain the CC BY-NC 4.0 attribution notice.

This first run transfers roughly 250 GB of model weights plus the dataset and
container layers, so it can take substantial time. The cell fails before the
large model download if a configured identity is malformed.


In [ ]:
download(CONFIG, WORKSPACE)

## Step 4: Prepare the data

- Remove empty text items and labels contaminated with reasoning or OCR instructions.
- Reserve 400 validation images using a fixed seed-42 hash split.
- Repeat training examples with 5–8 reactions twice and those with 9+ reactions three times.
- Check that training, validation, and benchmark images do not overlap.

Cleaned data goes into `prepared/`. The weighted training rows determine the
number of updates, using a global batch size of 16.


In [ ]:
RUN_DIR.mkdir(parents=True, exist_ok=True)
if (RUN_DIR / "training_started.json").exists():
    raise FileExistsError("This run has already started training. Choose a new RUN_ID.")
prepared = prepare_records(DATA, BENCHMARK, RUN_DIR / "prepared", VALIDATION_SIZE)
STEPS = SMOKE_STEPS or prepared["full_steps"]
if STEPS > prepared["full_steps"]:
    raise ValueError("Smoke steps must fit within one epoch of the supplied dataset")
print({k: v for k, v in prepared.items() if not k.endswith("sha256")})

### Check a training example

Compare the image's box numbers with the target JSON. This cell shows the full
prompt and cleaned label used for training.


In [ ]:
with (RUN_DIR / "prepared/train.jsonl").open() as stream:
    example = json.loads(next(stream))
print("Training example:", example["id"])
with PILImage.open(DATA / example["image"]) as image:
    display(image.convert("RGB"))
print("\nUser prompt (including the image placeholder):\n")
print(example["prompt"])
print("\nExpected JSON target:\n")
print(json.dumps(json.loads(example["target"]), indent=2, ensure_ascii=False))

## Step 5: Set up LoRA training

The support module writes the training entry point and merge script into the run
directory. The full AutoModel recipe is shown below.

- **Adapters:** rank 64, alpha 128, dropout 0; dense language layers only. Vision and projector weights stay frozen.
- **Training:** learning rate 2e-4, sequence length 6,144, global batch 16. Four GPUs each use batch 1 and four accumulation steps.
- **Memory:** BF16 model weights, FP32 optimizer state, FSDP2, and expert parallelism 4.

Each GPU checks which parameters are trainable before training starts.


In [ ]:
# The support module writes the two entry scripts used inside Docker.
CODE_DIR = RUN_DIR / "code"
write_runtime_scripts(CODE_DIR)

In [ ]:
RECIPE_YAML = r"""
recipe: FinetuneRecipeForVLM
step_scheduler:
  global_batch_size: 16
  local_batch_size: 1
  val_every_steps: 125
  num_epochs: 1
  save_checkpoint_every_epoch: false
dist_env:
  backend: cpu:gloo,cuda:nccl
  timeout_minutes: 30
seed: 42
model:
  _target_: nemo_automodel.NeMoAutoModelForImageTextToText.from_pretrained
  pretrained_model_name_or_path: ${oc.env:MODEL_PATH}
  trust_remote_code: true
  num_nextn_predict_layers: 0
  torch_dtype: torch.bfloat16
  backend:
    _target_: nemo_automodel.components.models.common.BackendConfig
    attn: sdpa
    linear: torch
    rms_norm: torch_fp32
    rope_fusion: false
    dispatcher: deepep
    experts: torch_mm
    fake_balanced_gate: false
    enable_hf_state_dict_adapter: true
processor:
  _target_: transformers.AutoProcessor.from_pretrained
  pretrained_model_name_or_path: ${oc.env:MODEL_PATH}
  trust_remote_code: true
  fix_mistral_regex: true
checkpoint:
  enabled: true
  checkpoint_dir: ${oc.env:CHECKPOINT_DIR}
  model_save_format: safetensors
  save_consolidated: false
  cpu_offload: true
  max_recent_checkpoints: 1
distributed:
  strategy: fsdp2
  tp_size: 1
  cp_size: 1
  pp_size: 1
  ep_size: 4
  sequence_parallel: false
  activation_checkpointing: true
  reshard_after_forward: true
  defer_fsdp_grad_sync: false
  offload_policy: null
  moe:
    reshard_after_forward: true
  multimodal:
    frozen_sharding: root
freeze_config:
  freeze_modules:
  - path: language_model.model.embed_tokens
  - path: language_model.lm_head
  freeze_vision_tower: true
  freeze_video_embedder: true
  freeze_audio_tower: true
  freeze_language_model: false
loss_fn:
  _target_: nemo_automodel.components.loss.masked_ce.MaskedCrossEntropy
dataset:
  _target_: train_entry.make_dataset
  path_or_dataset: ${oc.env:PREPARED_DIR}
  split: train
  image_root: ${oc.env:DATA_DIR}
dataloader:
  _target_: torchdata.stateful_dataloader.StatefulDataLoader
  num_workers: 0
  collate_fn:
    _target_: nemo_automodel.components.datasets.vlm.collate_fns.nemotron_omni_collate_fn
    max_length: 6144
  drop_last: true
validation_dataset:
  _target_: train_entry.make_dataset
  path_or_dataset: ${oc.env:PREPARED_DIR}
  split: validation
  image_root: ${oc.env:DATA_DIR}
validation_dataloader:
  _target_: torchdata.stateful_dataloader.StatefulDataLoader
  num_workers: 0
  collate_fn:
    _target_: nemo_automodel.components.datasets.vlm.collate_fns.nemotron_omni_collate_fn
    max_length: 6144
optimizer:
  _target_: nemo_automodel.components.optim.optimizer.FusedAdamConfig
  lr: 0.0002
  weight_decay: 0.01
  betas:
  - 0.9
  - 0.95
  master_weights: true
  master_weight_dtype: float32
lr_scheduler:
  init_lr: 0.0002
  max_lr: 0.0002
  min_lr: 0.0002
  lr_warmup_steps: 0
  lr_decay_style: constant
peft:
  _target_: nemo_automodel.components._peft.lora.PeftConfig
  match_all_linear: false
  exclude_modules:
  - '*vision_tower*'
  - '*vision_model*'
  - '*vision_projector*'
  - '*audio*'
  - '*sound*'
  - '*lm_head*'
  - '*mlp1*'
  - '*.out_proj'
  dim: 64
  alpha: 128
  dropout: 0.0
  use_triton: true
"""

recipe = yaml.safe_load(RECIPE_YAML)
recipe["step_scheduler"]["max_steps"] = STEPS
recipe["step_scheduler"]["ckpt_every_steps"] = STEPS
recipe["lr_scheduler"]["lr_decay_steps"] = STEPS
if SMOKE_STEPS:
    recipe["step_scheduler"]["val_every_steps"] = STEPS
(RUN_DIR / "train.yaml").write_text(yaml.safe_dump(recipe, sort_keys=False))
print("Updates:", STEPS, "— smoke test:", SMOKE_STEPS is not None)

## Step 6: Train on four GPUs

Run training and wait for it to finish. Follow progress in `logs/train.log` under
your run directory. A run ID can only be used once, to protect existing checkpoints.


In [ ]:
def container_path(path):
    return str(Path("/workspace") / Path(path).resolve().relative_to(WORKSPACE))


def docker_command(stage, arguments, gpu=False, extra_options=()):
    image = CONFIG["eval_image" if stage == "eval" else "train_image"]
    python = EVAL_PYTHON if stage == "eval" else TRAIN_PYTHON
    cache = WORKSPACE / "cache"
    (cache / "home").mkdir(parents=True, exist_ok=True)
    command = [
        "docker",
        "run",
        "--rm",
        "--network=host",
        "--ipc=host",
        "--ulimit",
        "memlock=-1",
        "--user",
        f"{os.getuid()}:{os.getgid()}",
        "--mount",
        f"type=bind,source={WORKSPACE},target=/workspace",
        "--workdir",
        "/workspace",
        "--entrypoint",
        python,
    ]
    if gpu:
        command += ["--gpus", "all"]
    if CPU_MEMORY_NODES:
        command += ["--cpuset-mems", CPU_MEMORY_NODES]
    environment = dict(
        PYTHONPATH=f"{container_path(ASSETS / 'Automodel')}:{container_path(CODE_DIR)}",
        RUN_DIR=container_path(RUN_DIR),
        MODEL_PATH=container_path(BASE_MODEL),
        DATA_DIR=container_path(DATA),
        PREPARED_DIR=container_path(RUN_DIR / "prepared"),
        CHECKPOINT_DIR=container_path(RUN_DIR / "checkpoints"),
        HF_HOME=container_path(cache / "huggingface"),
        HOME=container_path(cache / "home"),
        XDG_CACHE_HOME=container_path(cache),
        PYTHONUNBUFFERED="1",
        PYTHONDONTWRITEBYTECODE="1",
        OMP_NUM_THREADS="8",
        TOKENIZERS_PARALLELISM="false",
        PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
        VLLM_NO_USAGE_STATS="1",
    )
    environment.update(RUNTIME_ENV)
    if stage == "eval":
        environment["CUDA_VISIBLE_DEVICES"] = "0,1"
    for key, value in environment.items():
        command += ["--env", f"{key}={value}"]
    return command + list(extra_options) + [image] + list(map(str, arguments))


train_command = docker_command(
    "train",
    [
        "-m",
        "torch.distributed.run",
        "--standalone",
        "--nnodes=1",
        "--nproc-per-node=4",
        "--max-restarts=0",
        container_path(CODE_DIR / "train_entry.py"),
        container_path(RUN_DIR / "train.yaml"),
    ],
    gpu=True,
)

In [ ]:
# This marker prevents an accidental repeat from overwriting a partial/completed run.
with (RUN_DIR / "training_started.json").open("x") as stream:
    json.dump(
        dict(
            steps=STEPS, recipe_sha256=sha256(RUN_DIR / "train.yaml"), entry_sha256=sha256(CODE_DIR / "train_entry.py")
        ),
        stream,
    )
run_logged(train_command, RUN_DIR / "logs/train.log")
checkpoint = RUN_DIR / "checkpoints" / f"epoch_0_step_{STEPS - 1}" / "model"
assert (checkpoint / "adapter_model.safetensors").is_file(), "Final adapter checkpoint is missing"
audits = [json.loads((RUN_DIR / "audit" / f"rank{rank}.json").read_text()) for rank in range(4)]
assert all(a["status"] == "passed" for a in audits)
write_json(RUN_DIR / "training_complete.json", dict(steps=STEPS, smoke_test=SMOKE_STEPS is not None))
print("Training and all four trainability audits passed.")

## Step 7: Merge the adapter

Merge the learned adapters into the BF16 base model, one shard at a time on CPU.
The script checks that every adapter was applied and that all output shards are
complete. It runs in the training container.


In [ ]:
assert (RUN_DIR / "training_complete.json").is_file(), "Finish training first"
run_logged(docker_command("export", [container_path(CODE_DIR / "merge_adapter.py")]), RUN_DIR / "logs/export.log")
exported = json.loads((RUN_DIR / "export.json").read_text())
print(json.dumps(exported, indent=2))

## Step 8: Set up evaluation

The scorer reports the two metrics defined in the
[RxnCaption paper](https://arxiv.org/abs/2511.02384):

- **SoftMatch** ignores text, merges molecular conditions into reactants, and
  keeps products direction-aware.
- **HybridMatch** matches molecules by role, requires exact reactant/product
  text, and allows normalized edit distance up to 0.2 for condition text.

Malformed JSON counts as an empty prediction and remains in the score.
All scoring runs on CPU; vLLM handles inference on two GPUs. The response parser
and matching rules are in [rxncaption_support.py](rxncaption_support.py).


In [ ]:
# Load the reaction-specific scorer and serving lifecycle.
from rxncaption_support import evaluate_checkpoint

## Step 9: Evaluate the model

Start vLLM and score all 400 benchmark images using BF16, temperature 0, thinking
disabled, and at most 2,048 output tokens. Follow startup in `evaluation/server.log`.

The cell saves predictions and scores, then stops the serving container.
Failed inference requests are recorded and cause the evaluation to fail.


In [ ]:
results = evaluate_checkpoint(
    RUN_DIR,
    BENCHMARK,
    port=EVAL_PORT,
    docker_command=docker_command,
    container_path=container_path,
)

## Step 10: Read the results

Scores are F1 percentages; higher is better. Inspect `evaluation/raw.json` alongside
the images to understand mistakes. Checkpoints, logs, and metrics are in `RUN_DIR`.

Use a new run ID when changing the recipe, and keep the benchmark fixed.


In [ ]:
for name, metric in results["metrics"].items():
    print(f"{name:24s} F1 = {metric['f1']:.2f}%")
print("Images:", results["images"])
print("Invalid JSON:", len(results["invalid_json_images"]))
print("Smoke test:", results["smoke_test"])
print("Artifacts:", RUN_DIR)

### Reference results

Earlier one-epoch runs (1,171 updates) on GB200 GPUs, evaluated on the same
400 image IDs and box mapping. Scores are F1 (%) using the paper-defined metrics.
These used a pre-release checkpoint and have not been remeasured with the GA
model. Pillow-version differences can change PNG rasterization, so treat these as
reference measurements rather than guaranteed reproduced scores.

| Training | Hardware | SoftMatch | HybridMatch |
|---|---|---:|---:|
| LoRA | 1 node × 4 GPUs | 81.04 | 64.01 |
| Full SFT | 4 nodes × 4 GPUs | 85.73 | 69.86 |

LoRA used rank 64, learning rate 2e-4, and frozen vision weights. Full SFT used
learning rate 3e-6 and trainable vision weights. A five-step run checks the
workflow, not final model quality.
